<a href="https://colab.research.google.com/github/RafaelaMlucca/conformal-violence-prediction/blob/main/02_baseline_mapie_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — MAPIE baseline

Roda MAPIE (LAC e APS) como baseline de conformal prediction para os 3 desfechos.

**Mudança:** lê os arquivos .npz salvos pelo notebook 01 (em vez do pickle único).
A logística aceita matrizes esparsas direto, MAPIE também.

## 1. Setup

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!pip install -q mapie

In [16]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import json
from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from mapie.classification import SplitConformalClassifier

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
SAIDA = DRIVE / 'conformal_violence'

RANDOM_STATE = 42
ALPHA = 0.1
CONFIDENCE = 1 - ALPHA

## 2. Carregando dados

In [17]:
# 1. Remonta o Drive se necessário
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# 2. Verifica se os arquivos existem no Drive
from pathlib import Path
SAIDA = Path('/content/drive/MyDrive/projeto_violencia_mulher/conformal_violence')
print('Arquivos no Drive:')
for arq in sorted(SAIDA.iterdir()):
    if arq.is_file():
        print(f'  {arq.name}: {arq.stat().st_size / 1024 / 1024:.1f} MB')

Mounted at /content/drive
Arquivos no Drive:
  X_cal.npz: 15.0 MB
  X_test.npz: 15.7 MB
  X_train.npz: 35.0 MB
  cobertura_local_y_fisic.csv: 0.0 MB
  cobertura_local_y_psico.csv: 0.0 MB
  cobertura_local_y_sexu.csv: 0.0 MB
  dados_conformal.pkl: 3425.5 MB
  metadados.json: 0.0 MB
  resultados_baseline.pkl: 9.8 MB
  resultados_locart.pkl: 24.4 MB
  tabela_comparativa_final.csv: 0.0 MB
  y_cal.parquet: 0.3 MB
  y_test.parquet: 0.3 MB
  y_train.parquet: 0.4 MB


In [18]:
X_train = load_npz(SAIDA / 'X_train.npz')
X_cal   = load_npz(SAIDA / 'X_cal.npz')
X_test  = load_npz(SAIDA / 'X_test.npz')

y_train = pd.read_parquet(SAIDA / 'y_train.parquet')
y_cal   = pd.read_parquet(SAIDA / 'y_cal.parquet')
y_test  = pd.read_parquet(SAIDA / 'y_test.parquet')

with open(SAIDA / 'metadados.json') as f:
    meta = json.load(f)

ALVOS = meta['alvos']
feature_names = meta['feature_names']

print(f'Treino: {X_train.shape[0]:,} x {X_train.shape[1]}  (esparso)')
print(f'Cal:    {X_cal.shape[0]:,}  (esparso)')
print(f'Teste:  {X_test.shape[0]:,}  (esparso)')

Treino: 1,733,713 x 275  (esparso)
Cal:    743,021  (esparso)
Teste:  852,044  (esparso)


## 3. Treinando uma logística por desfecho


In [19]:
modelos = {}
for alvo in ALVOS:
    modelo = LogisticRegression(
        penalty='l1', solver='liblinear', C=1.0,
        max_iter=500, random_state=RANDOM_STATE
    )
    modelo.fit(X_train, y_train[alvo].values)
    modelos[alvo] = modelo
    print(f'{alvo}: treinado')

y_fisic: treinado
y_psico: treinado
y_sexu: treinado


## 4. Função auxiliar

In [20]:
def avaliar_conjuntos(y_verdadeiro, conjuntos):
    n = len(y_verdadeiro)
    cobertos = np.array([conjuntos[i, y_verdadeiro[i]] for i in range(n)])
    tamanhos = conjuntos.sum(axis=1)
    return {
        'cobertura': float(cobertos.mean()),
        'tamanho_medio': float(tamanhos.mean()),
        'pct_vazios': float((tamanhos == 0).mean()),
        'pct_unitarios': float((tamanhos == 1).mean()),
        'pct_duplos': float((tamanhos == 2).mean()),
    }

## 5. MAPIE com método LAC

In [21]:
resultados_lac = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    # converte só agora, dentro do loop
    X_cal_dense = X_cal.toarray()
    X_test_dense = X_test.toarray()

    mapie = SplitConformalClassifier(
        estimator=modelo,
        confidence_level=CONFIDENCE,
        conformity_score='lac',
        prefit=True,
        random_state=RANDOM_STATE,
    )
    mapie.conformalize(X_cal_dense, y_cal_alvo)

    _, conjuntos = mapie.predict_set(X_test_dense)
    conjuntos = conjuntos[:, :, 0]

    # libera memória imediatamente
    del X_cal_dense, X_test_dense

    res = avaliar_conjuntos(y_test_alvo, conjuntos)
    res['conjuntos'] = conjuntos
    resultados_lac[alvo] = res

    print(f"{alvo}: cob={res['cobertura']:.4f}, tam={res['tamanho_medio']:.4f}, vazios={res['pct_vazios']:.1%}")

y_fisic: cob=0.9065, tam=1.1183, vazios=0.0%
y_psico: cob=0.9062, tam=1.1549, vazios=0.0%
y_sexu: cob=0.8947, tam=0.9502, vazios=5.0%


## 6. MAPIE com método APS

In [22]:
resultados_aps = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    X_cal_dense = X_cal.toarray()
    X_test_dense = X_test.toarray()

    mapie = SplitConformalClassifier(
    estimator=modelo,
    confidence_level=CONFIDENCE,
    conformity_score='lac',
    prefit=True,
    random_state=RANDOM_STATE,)
    mapie.conformalize(X_cal_dense, y_cal_alvo)

    _, conjuntos = mapie.predict_set(
        X_test_dense,
        conformity_score_params={'include_last_label': True}
    )
    conjuntos = conjuntos[:, :, 0]

    del X_cal_dense, X_test_dense

    res = avaliar_conjuntos(y_test_alvo, conjuntos)
    res['conjuntos'] = conjuntos
    resultados_aps[alvo] = res

    print(f"{alvo}: cob={res['cobertura']:.4f}, tam={res['tamanho_medio']:.4f}, vazios={res['pct_vazios']:.1%}")

y_fisic: cob=0.9065, tam=1.1183, vazios=0.0%
y_psico: cob=0.9062, tam=1.1549, vazios=0.0%
y_sexu: cob=0.8947, tam=0.9502, vazios=5.0%


## 7. Tabela comparativa

In [23]:
linhas = []
for alvo in ALVOS:
    linhas.append({
        'desfecho': alvo, 'metodo': 'LAC',
        **{k: v for k, v in resultados_lac[alvo].items() if k != 'conjuntos'},
    })
    linhas.append({
        'desfecho': alvo, 'metodo': 'APS',
        **{k: v for k, v in resultados_aps[alvo].items() if k != 'conjuntos'},
    })

tabela = pd.DataFrame(linhas)
print(tabela.to_string(index=False))

desfecho metodo  cobertura  tamanho_medio  pct_vazios  pct_unitarios  pct_duplos
 y_fisic    LAC   0.906491       1.118269    0.000000       0.881731    0.118269
 y_fisic    APS   0.906491       1.118269    0.000000       0.881731    0.118269
 y_psico    LAC   0.906176       1.154893    0.000000       0.845107    0.154893
 y_psico    APS   0.906176       1.154893    0.000000       0.845107    0.154893
  y_sexu    LAC   0.894666       0.950211    0.049789       0.950211    0.000000
  y_sexu    APS   0.894666       0.950211    0.049789       0.950211    0.000000


## 8. Salvando

In [24]:
resultados_baseline = {
    'modelos': modelos,
    'lac': resultados_lac,
    'aps': resultados_aps,
    'alpha': ALPHA,
    'confidence': CONFIDENCE,
    'tabela_comparativa': tabela,
}

with open(SAIDA / 'resultados_baseline.pkl', 'wb') as f:
    pickle.dump(resultados_baseline, f)

print(f'Salvo em {SAIDA / "resultados_baseline.pkl"}')

Salvo em /content/drive/MyDrive/projeto_violencia_mulher/conformal_violence/resultados_baseline.pkl
